# Lab 12 Tasks and Solutions: Non-Linear Regression and Spatial Evaluation

**Total: 100 points.** Each task has a point value and an empty starter cell.

Data: California Housing (1990 Census, scikit-learn copy), 20,640 block groups. The target is `MedHouseVal`, median house value in units of 100,000 dollars. All random choices use `random_state=0` unless a task says otherwise, so your numbers should match the key. Grading is by method and reasoning, not by matching every digit. Spatial-fold numbers can differ slightly from the lecture slides, because `GroupKFold` assigns blocks to folds differently in this environment's older scikit-learn. Task 8 has no single correct answer.

## Setup

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn import cluster, ensemble, inspection, linear_model, metrics, model_selection, tree
from libpysal import weights
from esda import Moran

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| geopandas", gpd.__version__,
      "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

Python 3.8.16 | numpy 1.24.3 | pandas 2.0.2 | geopandas 0.13.2 | scikit-learn 1.2.2 | matplotlib 3.7.1


In [2]:
PURPLE, MAGENTA, BLUE, GRAY = "#7C3AED", "#DB2777", "#2563EB", "#6B6478"
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})


def rmse(y_true, y_pred):
    """Root mean squared error (scikit-learn 1.2.2 has no root_mean_squared_error)."""
    return float(np.sqrt(metrics.mean_squared_error(y_true, y_pred)))


def scores(model, X_tr, y_tr, X_te, y_te):
    """Train R2, test R2, test MAE, test RMSE of a fitted model."""
    p = model.predict(X_te)
    return dict(train_r2=model.score(X_tr, y_tr), test_r2=model.score(X_te, y_te),
                mae=metrics.mean_absolute_error(y_te, p), rmse=rmse(y_te, p))


def blank(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


def point_map(ax, x, y, values, cmap, vmin, vmax, title, label):
    """Scatter map of block groups (projected meters), no ticks, with a colorbar."""
    sc = ax.scatter(x, y, c=values, s=3, cmap=cmap, vmin=vmin, vmax=vmax, linewidths=0)
    ax.set_aspect("equal"); blank(ax); ax.set_title(title)
    cb = plt.colorbar(sc, ax=ax, shrink=0.7); cb.set_label(label)

In [3]:
df = pd.read_csv("data/california_housing.csv")
xy = gpd.GeoSeries(gpd.points_from_xy(df.Longitude, df.Latitude), crs=4326).to_crs(3310)   # California Albers, meters
df["x"], df["y"] = xy.x.values, xy.y.values
print(df.shape)
ANS = {}                                  # answers are stored here by the solution cells

(20640, 11)


In [4]:
F6 = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup"]
F8 = F6 + ["Latitude", "Longitude"]
y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = model_selection.train_test_split(df[F6], y, test_size=0.2, random_state=0)
print(X_train.shape, X_test.shape)

(16512, 6) (4128, 6)


## Task 1 (10 points): The baseline line

Fit `LinearRegression` on the six features `F6` using the split above. Report the train $R^2$, the test $R^2$, and the test MAE and RMSE (use the `scores` helper or compute them). Say in one sentence what the MAE means in dollars.

In [5]:
# your code here

## Task 2 (12 points): How deep should the tree grow?

Fit `DecisionTreeRegressor(max_depth=d, random_state=0)` for every depth from 1 to 15. Report the depth with the highest **test** $R^2$, its train and test $R^2$, and the train and test $R^2$ at depth 15. Explain in two sentences why the training $R^2$ keeps rising while the test $R^2$ does not.

In [7]:
# your code here

## Task 3 (12 points): Forest versus line

Fit `RandomForestRegressor(n_estimators=n, n_jobs=-1, random_state=0)` on `F6` with `n` equal to 10 and to 100. Report the test $R^2$ of each and of the line from Task 1, and the training $R^2$ of the 100-tree forest. In one sentence, say what the gap between its training and test $R^2$ tells you.

In [9]:
# your code here

## Task 4 (12 points): Tune gradient boosting

Use `GridSearchCV` with 5 folds on the **training** rows to tune `GradientBoostingRegressor(random_state=0)` over `learning_rate` in 0.03, 0.1, 0.3 and `max_depth` in 2, 4, 6. Report the best setting, its 5-fold cross-validated $R^2$, and the test $R^2$ of the refit best model. Why is the test set used only once, after the search?

In [11]:
# your code here

## Task 5 (14 points): Add latitude and longitude

Using the same split, fit a `LinearRegression` and a `RandomForestRegressor(n_jobs=-1, random_state=0)` on the eight features `F8`. Report the test $R^2$ of both and the change from the six-feature models (Tasks 1 and 3). Which model gains more from location, and why might that be?

In [13]:
# your code here

## Task 6 (16 points): Random folds versus spatial folds

Build 40 spatial blocks with `KMeans(n_clusters=40, n_init=4, random_state=0)` on the projected coordinates `df[["x", "y"]]`. For `RandomForestRegressor(n_jobs=-1, random_state=0)` on `F8`, compute the 5-fold cross-validated $R^2$ with (a) `KFold(5, shuffle=True, random_state=0)` and (b) `GroupKFold(5)` with the blocks as groups, using `cross_val_predict` and `metrics.r2_score` on all rows. Report both scores and the gap. Repeat for `LinearRegression`. Explain which score you would report for a new county the model has never seen, and why the line's gap is small.

In [15]:
# your code here

## Task 7 (12 points): Residual map and Moran's I

Using the spatially cross-validated forest predictions from Task 6, compute the residuals (actual minus predicted). Build row-standardized `libpysal.weights.KNN.from_array` weights with $k = 8$ on `df[["x", "y"]]`, and compute Moran's I of the residuals with `esda.Moran(..., permutations=99)` after `np.random.seed(0)`. Map the residuals with a colorbar and no ticks. Is there spatial structure left? Name one place in the map where the model clearly misses.

In [17]:
# your code here

## Task 8 (12 points): What does the forest use? (open-ended)

Hold out the first spatial fold (`train, test = next(GroupKFold(5).split(df[F8], y, groups=block))`), fit a forest on the rest, and compute `permutation_importance` on the held-out rows (`n_repeats=5`, `random_state=0`). Report the three most important features and their drops. Then write 3 to 5 sentences: what can you conclude about the model, and what can you **not** conclude about houses in California?

In [19]:
# your code here

In [20]:
train, test = next(model_selection.GroupKFold(5).split(df[F8], y, groups=block))
rf_f = ensemble.RandomForestRegressor(n_jobs=-1, random_state=0).fit(df[F8].iloc[train], y.iloc[train])
imp = inspection.permutation_importance(rf_f, df[F8].iloc[test], y.iloc[test], n_repeats=5, random_state=0, n_jobs=-1)
top = sorted(zip(F8, imp.importances_mean), key=lambda p: -p[1])
for name, drop in top:
    print(f"{name:10s} {drop:.3f}")
ANS["T8"] = dict(top=[(n, float(d)) for n, d in top[:3]], fold_r2=float(rf_f.score(df[F8].iloc[test], y.iloc[test])))

MedInc     0.566
Latitude   0.383
AveOccup   0.246
Longitude  0.121
HouseAge   0.034
AveRooms   0.025
Population 0.008
AveBedrms  0.006


**Answers (one good answer):** The top three features are MedInc (drop 0.566), Latitude (0.383) and AveOccup (0.246) on a fold where the forest scores $R^2$ = 0.714. Conclusion about the model: it relies mostly on income and location, and shuffling any of the other features costs it very little on unseen areas. What cannot be concluded: that income or latitude *cause* higher values, since importance measures what this model uses, not why prices differ. Features that move together (rooms, bedrooms, occupancy) can share importance, and latitude may stand in for coast, climate or distance to big cities. The data are from 1990 and values are capped at 500,001 dollars. A weak answer reads importance as causation or reports the ranking with no caveat.

## Optional (not graded)

Replace the KMeans blocks by 50 km grid squares and repeat the forest comparison of Task 6. Does the gap between random and spatial folds grow or shrink?